In [ ]:
#!/usr/bin/env python
# =============================================================================
# FLIPKART GRiD 6.0 — AI-DRIVEN PARKING INTELLIGENCE
# Problem: Poor Visibility on Parking-Induced Congestion
# Goal: Detect illegal parking hotspots & quantify traffic flow impact
# =============================================================================

# ======================= CELL 1: INSTALL DEPENDENCIES ========================
import subprocess, sys
for pkg in ["hdbscan", "folium", "xgboost", "kagglehub[pandas-datasets]"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# ======================= CELL 2: IMPORT LIBRARIES ============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import re
import os
from datetime import datetime

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, silhouette_score,
    accuracy_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score
)
from xgboost import XGBClassifier
import hdbscan

# Visualization
import folium
from folium.plugins import HeatMap

warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print("=" * 70)
print("  FLIPKART GRiD 6.0 — PARKING INTELLIGENCE PIPELINE")
print("  AI-Driven Illegal Parking Hotspot Detection")
print("=" * 70)
print("\n✅ All libraries loaded successfully!")

# ======================= CELL 3: LOAD DATA ===================================
# ======================= CELL 3: LOAD DATA ===================================
import os

print("Loading dataset from Kaggle input directory...")

# Standard Kaggle path format when a dataset is attached
# Usually: /kaggle/input/<dataset-slug>/<filename>
DATA_PATH = "/kaggle/input/flipkart-gridlock-r2-theme-1/jan to may police violation_anonymized791b166.csv"

# Fallback to search if the exact path is slightly different
if not os.path.exists(DATA_PATH):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if filename.endswith('.csv'):
                DATA_PATH = os.path.join(dirname, filename)
                print(f"Found data at: {DATA_PATH}")
                break

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"\n📊 Dataset Shape: {df.shape}")
print(f"📋 Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")


# ======================= CELL 4: DATASET OVERVIEW ============================
print("\n" + "=" * 70)
print("  SECTION 1: DATASET OVERVIEW")
print("=" * 70)

print("\n🔍 First 5 Rows:")
display(df.head()) if 'display' in dir() else print(df.head())

print("\n📈 Data Types:")
display(df.dtypes) if 'display' in dir() else print(df.dtypes)

print("\n📉 Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
missing_report = missing_df[missing_df['Missing'] > 0].sort_values('Percentage', ascending=False)
display(missing_report) if 'display' in dir() else print(missing_report)

print("\n📊 Basic Statistics:")
display(df.describe()) if 'display' in dir() else print(df.describe())

# ======================= CELL 5: DATA CLEANING ===============================
print("\n" + "=" * 70)
print("  SECTION 2: DATA CLEANING")
print("=" * 70)

initial_len = len(df)

# Drop columns that are 100% null
drop_cols = ['description', 'closed_datetime', 'action_taken_timestamp']
df.drop(columns=drop_cols, inplace=True, errors='ignore')
print(f"   ✅ Dropped 100% null columns: {drop_cols}")

# Drop mostly-null timestamp column
df.drop(columns=['data_sent_to_scita_timestamp'], inplace=True, errors='ignore')
print(f"   ✅ Dropped 85%+ null column: data_sent_to_scita_timestamp")

# Fill missing values
df['center_code'] = df['center_code'].fillna(-1).astype(int)
df['police_station'] = df['police_station'].fillna('Unknown')
df['junction_name'] = df['junction_name'].fillna('No Junction')
df['location'] = df['location'].fillna('Unknown')
df['created_by_id'] = df['created_by_id'].fillna('Unknown')
df['validation_status'] = df['validation_status'].fillna('not_validated')
df['updated_vehicle_type'] = df['updated_vehicle_type'].fillna(df['vehicle_type'])

df = df.dropna(subset=['latitude', 'longitude'])
print(f"   ✅ Dropped {initial_len - len(df)} rows with missing lat/lng")

# Geo-filter to Bengaluru bounding box
df = df[
    (df['latitude'] >= 12.7) & (df['latitude'] <= 13.2) &
    (df['longitude'] >= 77.3) & (df['longitude'] <= 77.9)
]
print(f"   ✅ After geo-filtering (Bengaluru bounds): {len(df)} rows")

# Parse datetime columns
datetime_cols = ['created_datetime', 'modified_datetime', 'validation_timestamp']
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
        print(f"   ✅ Parsed '{col}' -> datetime")

# Drop duplicates
dupes = df.duplicated().sum()
df = df.drop_duplicates()
print(f"   ✅ Removed {dupes} duplicate rows")

print(f"\n✅ Clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

# ======================= CELL 6: PARSE VIOLATION TYPES =======================
print("\n" + "=" * 70)
print("  SECTION 3: VIOLATION TYPE PARSING")
print("=" * 70)

def parse_violation_list(val):
    """Parse the violation_type column which contains JSON-like lists."""
    if pd.isna(val):
        return []
    try:
        parsed = json.loads(val)
        if isinstance(parsed, list):
            return parsed
        return [str(parsed)]
    except (json.JSONDecodeError, TypeError):
        matches = re.findall(r'"([^"]*)"', str(val))
        if matches:
            return matches
        return [str(val).strip()]

def parse_offence_codes(val):
    """Parse the offence_code column which contains JSON-like lists."""
    if pd.isna(val):
        return []
    try:
        parsed = json.loads(str(val))
        if isinstance(parsed, list):
            return [int(x) for x in parsed]
        return [int(parsed)]
    except (json.JSONDecodeError, TypeError, ValueError):
        codes = re.findall(r'\d+', str(val))
        return [int(c) for c in codes]

# Apply parsing
df['violation_list'] = df['violation_type'].apply(parse_violation_list)
df['offence_code_list'] = df['offence_code'].apply(parse_offence_codes)
df['num_violations'] = df['violation_list'].apply(len)
df['primary_violation'] = df['violation_list'].apply(lambda x: x[0] if len(x) > 0 else 'UNKNOWN')

# Check unique violation types
all_violations = [v for vlist in df['violation_list'] for v in vlist]
print("📋 Unique Violation Types:")
for v, count in pd.Series(all_violations).value_counts().items():
    print(f"   • {v}: {count:,}")

# ======================= CELL 7: TEMPORAL FEATURE ENGINEERING ================
print("\n" + "=" * 70)
print("  SECTION 4: TEMPORAL FEATURE ENGINEERING")
print("=" * 70)

# Convert UTC to IST
df['created_datetime'] = df['created_datetime'].dt.tz_convert('Asia/Kolkata')
print("   ✅ Converted timestamps from UTC -> IST")

# Extract temporal features
df['hour'] = df['created_datetime'].dt.hour.fillna(0).astype(int)
df['day_of_week'] = df['created_datetime'].dt.dayofweek.fillna(0).astype(int)
df['day_name'] = df['created_datetime'].dt.day_name().fillna('Unknown')
df['month'] = df['created_datetime'].dt.month.fillna(0).astype(int)
df['month_name'] = df['created_datetime'].dt.month_name().fillna('Unknown')
df['date'] = df['created_datetime'].dt.date
df['week_of_year'] = df['created_datetime'].dt.strftime('%W').fillna('0').astype(int)
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Time periods
def get_time_period(hour):
    if 6 <= hour < 10:
        return 'Morning Rush'
    elif 10 <= hour < 14:
        return 'Midday'
    elif 14 <= hour < 18:
        return 'Afternoon Rush'
    elif 18 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

df['time_period'] = df['hour'].apply(get_time_period)
print("   ✅ Created: hour, day_of_week, day_name, month, is_weekend, time_period")

print(f"\n📊 Time Period Distribution:")
display(df['time_period'].value_counts()) if 'display' in dir() else print(df['time_period'].value_counts())

# ======================= CELL 8: SPATIAL FEATURE ENGINEERING =================
print("\n" + "=" * 70)
print("  SECTION 5: SPATIAL FEATURE ENGINEERING")
print("=" * 70)

# Grid cells (~111m per 0.001 degree)
df['lat_grid'] = (df['latitude'] * 1000).round() / 1000
df['lng_grid'] = (df['longitude'] * 1000).round() / 1000
df['grid_cell'] = df['lat_grid'].astype(str) + '_' + df['lng_grid'].astype(str)

# Violations per grid cell
grid_counts = df.groupby('grid_cell').size().reset_index(name='grid_violation_count')
df = df.merge(grid_counts, on='grid_cell', how='left')

# Violations per police station
station_counts = df.groupby('police_station').size().reset_index(name='station_violation_count')
df = df.merge(station_counts, on='police_station', how='left')

# Daily density per grid cell
daily_grid = df.groupby(['date', 'grid_cell']).size().reset_index(name='daily_grid_violations')
df = df.merge(daily_grid, on=['date', 'grid_cell'], how='left')

# Hourly density per grid cell
hourly_grid = df.groupby(['date', 'grid_cell', 'hour']).size().reset_index(name='hourly_grid_violations')
df = df.merge(hourly_grid, on=['date', 'grid_cell', 'hour'], how='left')

print("   ✅ Created: grid_cell, grid_violation_count, station_violation_count")
print("   ✅ Created: daily_grid_violations, hourly_grid_violations")
print(f"\n   Total unique grid cells: {df['grid_cell'].nunique():,}")
print(f"   Total unique police stations: {df['police_station'].nunique()}")

# ======================= CELL 9: EDA — VIOLATION PATTERNS ====================
print("\n" + "=" * 70)
print("  SECTION 6: EDA — VIOLATION PATTERNS")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Parking Violation Patterns — Bengaluru', fontsize=16, fontweight='bold', y=1.02)

# Plot 1: Violations by Hour
hourly = df.groupby('hour').size()
axes[0, 0].bar(hourly.index, hourly.values, color=sns.color_palette("viridis", len(hourly)), edgecolor='black', linewidth=0.5)
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Number of Violations')
axes[0, 0].set_title('Violations by Hour of Day')
axes[0, 0].set_xticks(range(0, 24))
axes[0, 0].axvspan(7, 10, alpha=0.15, color='red', label='Morning Rush')
axes[0, 0].axvspan(17, 20, alpha=0.15, color='orange', label='Evening Rush')
axes[0, 0].legend()

# Plot 2: Violations by Day of Week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df.groupby('day_name').size().reindex(day_order)
colors = ['#2ecc71' if d in ['Saturday', 'Sunday'] else '#3498db' for d in day_order]
axes[0, 1].bar(range(7), daily.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels([d[:3] for d in day_order])
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Number of Violations')
axes[0, 1].set_title('Violations by Day of Week (Green = Weekend)')

# Plot 3: Top 10 Violation Types
top_violations = pd.Series(all_violations).value_counts().head(10)
axes[1, 0].barh(range(len(top_violations)), top_violations.values, color=sns.color_palette("magma", len(top_violations)))
axes[1, 0].set_yticks(range(len(top_violations)))
axes[1, 0].set_yticklabels(top_violations.index, fontsize=9)
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Top 10 Violation Types')
axes[1, 0].invert_yaxis()

# Plot 4: Weekend vs Weekday
weekend_data = df.groupby(['is_weekend', 'time_period']).size().unstack(fill_value=0)
weekend_data.index = ['Weekday', 'Weekend']
weekend_data.plot(kind='bar', ax=axes[1, 1], edgecolor='black', linewidth=0.5)
axes[1, 1].set_title('Violations by Time Period: Weekday vs Weekend')
axes[1, 1].set_xlabel('')
axes[1, 1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)
axes[1, 1].legend(title='Time Period', fontsize=8)

plt.tight_layout()
plt.savefig('eda_violation_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: eda_violation_patterns.png")

# ======================= CELL 10: EDA — HEATMAP HOUR x DAY ==================
print("\n" + "=" * 70)
print("  SECTION 7: EDA — HEATMAP (Hour x Day of Week)")
print("=" * 70)

pivot = df.pivot_table(index='day_name', columns='hour', values='id', aggfunc='count')
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, fmt='d', linewidths=0.5,
            cbar_kws={'label': 'Number of Violations'}, ax=ax)
ax.set_title('Violation Heatmap: Hour of Day vs Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('eda_heatmap_hour_day.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: eda_heatmap_hour_day.png")

# ======================= CELL 11: EDA — MONTHLY TRENDS ======================
print("\n" + "=" * 70)
print("  SECTION 8: EDA — MONTHLY TRENDS")
print("=" * 70)

monthly = df.groupby([df['created_datetime'].dt.to_period('M')]).size()
fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(kind='bar', color=sns.color_palette("coolwarm", len(monthly)), edgecolor='black', ax=ax)
ax.set_title('Monthly Violation Trends', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Violations')
ax.set_xticklabels([str(x) for x in monthly.index], rotation=45)
for i, v in enumerate(monthly.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: eda_monthly_trend.png")

# ======================= CELL 12: EDA — VEHICLE TYPES & JUNCTIONS ===========
print("\n" + "=" * 70)
print("  SECTION 9: EDA — VEHICLE TYPES & JUNCTIONS")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vehicle type distribution
vtype = df['vehicle_type'].value_counts().head(10)
axes[0].pie(vtype.values, labels=vtype.index, autopct='%1.1f%%',
            colors=sns.color_palette("Set2", len(vtype)), startangle=90)
axes[0].set_title('Top Vehicle Types Involved', fontsize=12, fontweight='bold')

# Junction vs No Junction
junction_data = df['junction_name'].apply(lambda x: 'At Junction' if x != 'No Junction' else 'No Junction')
junc_counts = junction_data.value_counts()
axes[1].bar(junc_counts.index, junc_counts.values, color=['#e74c3c', '#3498db'], edgecolor='black')
axes[1].set_title('Violations: Junction vs Non-Junction', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(junc_counts.values):
    axes[1].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_vehicle_types.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: eda_vehicle_types.png")

# ======================= CELL 13: EDA — TOP HOTSPOT LOCATIONS ================
print("\n" + "=" * 70)
print("  SECTION 10: EDA — TOP 20 HOTSPOT LOCATIONS")
print("=" * 70)

top_locations = df.groupby(['police_station']).agg(
    total_violations=('id', 'count'),
    unique_vehicles=('vehicle_number', 'nunique'),
    unique_grid_cells=('grid_cell', 'nunique'),
    avg_daily_violations=('daily_grid_violations', 'mean')
).sort_values('total_violations', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(top_locations)), top_locations['total_violations'].values,
               color=sns.color_palette("rocket", len(top_locations)))
ax.set_yticks(range(len(top_locations)))
ax.set_yticklabels(top_locations.index, fontsize=10)
ax.set_xlabel('Total Violations', fontsize=12)
ax.set_title('Top 20 Police Station Areas by Violation Count', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(top_locations['total_violations'].values):
    ax.text(v + 200, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('eda_top_hotspots.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Top 20 Hotspot Statistics:")
display(top_locations) if 'display' in dir() else print(top_locations)
print("   ✅ Saved: eda_top_hotspots.png")

# ======================= CELL 14: DBSCAN HOTSPOT DETECTION ===================
print("\n" + "=" * 70)
print("  SECTION 11: HOTSPOT DETECTION (DBSCAN Spatial Clustering)")
print("=" * 70)

# Use grid-level aggregation for efficiency (avoids running DBSCAN on 300K rows)
grid_agg = df.groupby('grid_cell').agg(
    lat=('latitude', 'mean'),
    lng=('longitude', 'mean'),
    violation_count=('id', 'count'),
    unique_vehicles=('vehicle_number', 'nunique'),
    avg_hour=('hour', 'mean'),
    num_days=('date', 'nunique')
).reset_index()

print(f"   Grid cells for clustering: {len(grid_agg):,}")

# Standardize coordinates for DBSCAN
coords = grid_agg[['lat', 'lng']].values

# DBSCAN — eps ~0.002 (~200m), min_samples=3
# Using haversine would be ideal but euclidean on small lat/lng deltas is fine for Bengaluru
from sklearn.preprocessing import StandardScaler
coords_scaled = StandardScaler().fit_transform(coords)

db = DBSCAN(eps=0.3, min_samples=3)
grid_agg['cluster'] = db.fit_predict(coords_scaled)

n_clusters = len(set(grid_agg['cluster'])) - (1 if -1 in grid_agg['cluster'].values else 0)
n_noise = (grid_agg['cluster'] == -1).sum()

print(f"   ✅ DBSCAN Clusters found: {n_clusters}")
print(f"   ✅ Noise points: {n_noise}")

# Cluster statistics
cluster_stats = grid_agg[grid_agg['cluster'] != -1].groupby('cluster').agg(
    num_grid_cells=('grid_cell', 'count'),
    total_violations=('violation_count', 'sum'),
    center_lat=('lat', 'mean'),
    center_lng=('lng', 'mean'),
    avg_daily_activity=('num_days', 'mean')
).sort_values('total_violations', ascending=False)

print(f"\n📊 Top 10 Clusters by Violation Count:")
display(cluster_stats.head(10)) if 'display' in dir() else print(cluster_stats.head(10))

# ======================= CELL 15: HDBSCAN HOTSPOT DETECTION ==================
print("\n" + "=" * 70)
print("  SECTION 12: HDBSCAN HOTSPOT DETECTION (Density-Based)")
print("=" * 70)

# HDBSCAN for better density-based clustering
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3, metric='euclidean')
grid_agg['hdbscan_cluster'] = clusterer.fit_predict(coords_scaled)
grid_agg['hdbscan_prob'] = clusterer.probabilities_

n_hclusters = len(set(grid_agg['hdbscan_cluster'])) - (1 if -1 in grid_agg['hdbscan_cluster'].values else 0)
print(f"   ✅ HDBSCAN Clusters found: {n_hclusters}")
print(f"   ✅ Noise points: {(grid_agg['hdbscan_cluster'] == -1).sum()}")

# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# DBSCAN
scatter1 = axes[0].scatter(grid_agg['lng'], grid_agg['lat'],
                           c=grid_agg['cluster'], cmap='tab20',
                           s=grid_agg['violation_count'] * 0.5, alpha=0.6, edgecolors='black', linewidth=0.3)
axes[0].set_title(f'DBSCAN Clusters ({n_clusters} clusters)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# HDBSCAN
scatter2 = axes[1].scatter(grid_agg['lng'], grid_agg['lat'],
                           c=grid_agg['hdbscan_cluster'], cmap='tab20',
                           s=grid_agg['violation_count'] * 0.5, alpha=0.6, edgecolors='black', linewidth=0.3)
axes[1].set_title(f'HDBSCAN Clusters ({n_hclusters} clusters)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.suptitle('Spatial Clustering of Parking Violation Hotspots — Bengaluru', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('clustering_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: clustering_comparison.png")

# ======================= CELL 16: HOTSPOT ANALYSIS ===========================
print("\n" + "=" * 70)
print("  SECTION 13: HOTSPOT DEEP ANALYSIS")
print("=" * 70)

# Merge cluster labels back to main dataframe
grid_cluster_map = grid_agg[['grid_cell', 'cluster', 'hdbscan_cluster', 'hdbscan_prob']].copy()
df = df.merge(grid_cluster_map, on='grid_cell', how='left')

# Identify top hotspot clusters
top_clusters = cluster_stats.head(10)
print("\n🔥 TOP 10 HOTSPOT CLUSTERS:")
print("-" * 60)
for idx, row in top_clusters.iterrows():
    print(f"  Cluster {idx}: {int(row['total_violations']):,} violations | "
          f"{int(row['num_grid_cells'])} grid cells | "
          f"Center: ({row['center_lat']:.4f}, {row['center_lng']:.4f})")

# Hotspot characterization
print("\n📋 Hotspot vs Non-Hotspot Comparison:")
df['is_hotspot'] = (df['cluster'] != -1).astype(int)
comparison = df.groupby('is_hotspot').agg(
    total_violations=('id', 'count'),
    unique_locations=('grid_cell', 'nunique'),
    avg_hourly_density=('hourly_grid_violations', 'mean'),
    pct_weekend=('is_weekend', 'mean'),
    avg_num_violations=('num_violations', 'mean')
).round(3)
comparison.index = ['Non-Hotspot', 'Hotspot']
display(comparison) if 'display' in dir() else print(comparison)

# ======================= CELL 17: FOLIUM INTERACTIVE HEATMAP =================
print("\n" + "=" * 70)
print("  SECTION 14: INTERACTIVE VIOLATION HEATMAP")
print("=" * 70)

# Create heatmap data
heat_data = df[['latitude', 'longitude']].values.tolist()

# Center on Bengaluru
m = folium.Map(location=[12.9716, 77.5946], zoom_start=12, tiles='CartoDB dark_matter')

# Add violation heatmap
HeatMap(heat_data, radius=8, blur=10, max_zoom=13, gradient={
    0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1.0: 'red'
}).add_to(m)

# Add top cluster centers as markers
for idx, row in top_clusters.head(5).iterrows():
    folium.CircleMarker(
        location=[row['center_lat'], row['center_lng']],
        radius=12,
        color='red',
        fill=True,
        fill_opacity=0.8,
        popup=f"Hotspot Cluster {idx}<br>Violations: {int(row['total_violations']):,}"
    ).add_to(m)

m.save('violation_heatmap.html')
print("   ✅ Saved: violation_heatmap.html")
print("   📍 Top 5 hotspot clusters marked with red circles")
try:
    display(m)
except:
    print("   (Open violation_heatmap.html in browser to view)")

# ======================= CELL 18: CONGESTION IMPACT SCORE ====================
print("\n" + "=" * 70)
print("  SECTION 15: CONGESTION IMPACT SCORE (CIS) ENGINEERING")
print("=" * 70)

# Severity weights for violation types
violation_severity = {
    'WRONG PARKING': 3,
    'NO PARKING': 4,
    'PARKING IN A MAIN ROAD': 5,
    'DOUBLE PARKING': 5,
    'PARKING ON FOOTPATH': 4,
    'PARKING NEAR ROAD CROSSING': 5,
    'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS': 5,
    'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE': 4,
    'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC': 4,
    'PARKING OTHER THAN BUS STOP': 3,
    'H T V PROHIBITED': 4,
    'DEFECTIVE NUMBER PLATE': 1,
    'REFUSE TO GO FOR HIRE': 1,
    'USING BLACK FILM/OTHER MATERIALS': 1,
    'DEMANDING EXCESS FARE': 1,
    'WITHOUT SIDE MIRROR': 1,
    'AGAINST ONE WAY/NO ENTRY': 3,
    'VIOLATING LANE DISIPLINE': 3,
}

def get_severity(violation_list):
    if not violation_list:
        return 1
    return max(violation_severity.get(v, 2) for v in violation_list)

df['violation_severity'] = df['violation_list'].apply(get_severity)

# Time multiplier — rush hour violations impact traffic more
time_multiplier = {
    'Morning Rush': 1.5,
    'Afternoon Rush': 1.3,
    'Midday': 1.0,
    'Evening': 1.2,
    'Night': 0.5
}
df['time_multiplier'] = df['time_period'].map(time_multiplier)

# Junction multiplier — violations at junctions cause more congestion
df['junction_multiplier'] = df['junction_name'].apply(lambda x: 1.8 if x != 'No Junction' else 1.0)

# Vehicle size multiplier — larger vehicles block more road
vehicle_size = {
    'CAR': 1.0, 'SCOOTER': 0.5, 'MOTOR CYCLE': 0.5, 'AUTO': 0.7,
    'TEMPO': 1.5, 'LORRY': 2.0, 'BUS': 2.0, 'TANKER': 2.0,
    'MAXI-CAB': 1.3, 'TRACTOR': 1.5, 'MINI BUS': 1.5
}
df['vehicle_size_mult'] = df['vehicle_type'].map(vehicle_size).fillna(1.0)

# Density factor — normalized grid violation density
scaler = MinMaxScaler()
df['density_factor'] = scaler.fit_transform(df[['grid_violation_count']]) + 0.1

# CONGESTION IMPACT SCORE
df['congestion_impact_score'] = (
    df['violation_severity'] *
    df['time_multiplier'] *
    df['junction_multiplier'] *
    df['vehicle_size_mult'] *
    df['density_factor'] *
    df['num_violations']
).round(3)

print("   ✅ Components: violation_severity, time_multiplier, junction_multiplier,")
print("                  vehicle_size_mult, density_factor, num_violations")
print(f"\n📊 CIS Statistics:")
print(df['congestion_impact_score'].describe())

# ======================= CELL 19: CIS VISUALIZATION ==========================
print("\n" + "=" * 70)
print("  SECTION 16: CONGESTION IMPACT SCORE VISUALIZATION")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# CIS Distribution
axes[0].hist(df['congestion_impact_score'], bins=50, color='coral', edgecolor='black', alpha=0.8)
axes[0].set_title('CIS Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Congestion Impact Score')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['congestion_impact_score'].quantile(0.75), color='red', linestyle='--', label='75th percentile')
axes[0].axvline(df['congestion_impact_score'].quantile(0.90), color='darkred', linestyle='--', label='90th percentile')
axes[0].legend()

# CIS by Time Period
cis_time = df.groupby('time_period')['congestion_impact_score'].mean().sort_values(ascending=False)
axes[1].bar(range(len(cis_time)), cis_time.values, color=sns.color_palette("YlOrRd", len(cis_time)), edgecolor='black')
axes[1].set_xticks(range(len(cis_time)))
axes[1].set_xticklabels(cis_time.index, rotation=30, fontsize=9)
axes[1].set_title('Average CIS by Time Period', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg CIS')

# CIS by Vehicle Type (top 8)
cis_vehicle = df.groupby('vehicle_type')['congestion_impact_score'].mean().sort_values(ascending=False).head(8)
axes[2].barh(range(len(cis_vehicle)), cis_vehicle.values, color=sns.color_palette("RdYlGn_r", len(cis_vehicle)))
axes[2].set_yticks(range(len(cis_vehicle)))
axes[2].set_yticklabels(cis_vehicle.index, fontsize=10)
axes[2].set_title('Average CIS by Vehicle Type', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Avg CIS')
axes[2].invert_yaxis()

plt.suptitle('Congestion Impact Score Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cis_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: cis_analysis.png")

# ======================= CELL 20: CIS HEATMAP ON MAP =========================
print("\n" + "=" * 70)
print("  SECTION 17: CIS SPATIAL HEATMAP")
print("=" * 70)

# Aggregate CIS per grid cell
cis_grid = df.groupby('grid_cell').agg(
    lat=('latitude', 'mean'),
    lng=('longitude', 'mean'),
    avg_cis=('congestion_impact_score', 'mean'),
    total_violations=('id', 'count')
).reset_index()

# Weighted heatmap by CIS
cis_heat = cis_grid[['lat', 'lng', 'avg_cis']].values.tolist()

m2 = folium.Map(location=[12.9716, 77.5946], zoom_start=12, tiles='CartoDB positron')
HeatMap(cis_heat, radius=10, blur=12, max_zoom=13, gradient={
    0.2: 'green', 0.4: 'yellow', 0.6: 'orange', 0.8: 'red', 1.0: 'darkred'
}).add_to(m2)

m2.save('cis_heatmap.html')
print("   ✅ Saved: cis_heatmap.html (weighted by Congestion Impact Score)")
try:
    display(m2)
except:
    print("   (Open cis_heatmap.html in browser to view)")

# ======================= CELL 21: ML FEATURE PREPARATION =====================
print("\n" + "=" * 70)
print("  SECTION 18: ML FEATURE PREPARATION")
print("=" * 70)

# TARGET: Is this a HIGH-IMPACT violation? (top 25% by CIS)
cis_threshold = df['congestion_impact_score'].quantile(0.75)
df['high_impact'] = (df['congestion_impact_score'] >= cis_threshold).astype(int)

print(f"   CIS Threshold (75th percentile): {cis_threshold:.3f}")
print(f"   High Impact violations: {df['high_impact'].sum():,} ({df['high_impact'].mean()*100:.1f}%)")
print(f"   Low Impact violations:  {(1 - df['high_impact']).sum():,}")

# Select features for ML
le_vehicle = LabelEncoder()
le_station = LabelEncoder()
le_time = LabelEncoder()
le_violation = LabelEncoder()
le_junction = LabelEncoder()

df['vehicle_type_enc'] = le_vehicle.fit_transform(df['vehicle_type'].astype(str))
df['police_station_enc'] = le_station.fit_transform(df['police_station'].astype(str))
df['time_period_enc'] = le_time.fit_transform(df['time_period'].astype(str))
df['primary_violation_enc'] = le_violation.fit_transform(df['primary_violation'].astype(str))
df['junction_enc'] = le_junction.fit_transform(df['junction_name'].apply(
    lambda x: 'Junction' if x != 'No Junction' else 'No Junction'
).astype(str))

feature_cols = [
    'hour', 'day_of_week', 'month', 'is_weekend',
    'vehicle_type_enc', 'police_station_enc', 'time_period_enc',
    'primary_violation_enc', 'junction_enc',
    'num_violations', 'violation_severity',
    'grid_violation_count', 'station_violation_count',
    'daily_grid_violations', 'hourly_grid_violations',
    'latitude', 'longitude',
    'vehicle_size_mult', 'density_factor'
]

X = df[feature_cols].copy()
y = df['high_impact'].copy()

# Handle any remaining NaN
X = X.fillna(0)

print(f"\n   Feature matrix shape: {X.shape}")
print(f"   Features: {feature_cols}")

# ======================= CELL 22: TRAIN-TEST SPLIT ===========================
print("\n" + "=" * 70)
print("  SECTION 19: TRAIN-TEST SPLIT & SCALING")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train)
X_test_scaled = scaler_ml.transform(X_test)

print(f"   Train set: {X_train.shape[0]:,} samples")
print(f"   Test set:  {X_test.shape[0]:,} samples")
print(f"   Target distribution (train): {y_train.value_counts().to_dict()}")
print(f"   Target distribution (test):  {y_test.value_counts().to_dict()}")

# ======================= CELL 23: RANDOM FOREST ==============================
print("\n" + "=" * 70)
print("  SECTION 20: MODEL 1 — RANDOM FOREST CLASSIFIER")
print("=" * 70)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"   Accuracy:  {rf_acc:.4f}")
print(f"   F1 Score:  {rf_f1:.4f}")
print(f"   ROC AUC:   {rf_auc:.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, rf_pred, target_names=['Low Impact', 'High Impact']))

# ======================= CELL 24: XGBOOST ====================================
print("\n" + "=" * 70)
print("  SECTION 21: MODEL 2 — XGBOOST CLASSIFIER")
print("=" * 70)

# Calculate scale_pos_weight for imbalanced data
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    tree_method='hist'  # GPU: use 'gpu_hist' if available
)
xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_prob)

print(f"   Accuracy:  {xgb_acc:.4f}")
print(f"   F1 Score:  {xgb_f1:.4f}")
print(f"   ROC AUC:   {xgb_auc:.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Low Impact', 'High Impact']))

# ======================= CELL 25: GRADIENT BOOSTING ==========================
print("\n" + "=" * 70)
print("  SECTION 22: MODEL 3 — GRADIENT BOOSTING CLASSIFIER")
print("=" * 70)

gb_model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_prob = gb_model.predict_proba(X_test_scaled)[:, 1]

gb_acc = accuracy_score(y_test, gb_pred)
gb_f1 = f1_score(y_test, gb_pred)
gb_auc = roc_auc_score(y_test, gb_prob)

print(f"   Accuracy:  {gb_acc:.4f}")
print(f"   F1 Score:  {gb_f1:.4f}")
print(f"   ROC AUC:   {gb_auc:.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, gb_pred, target_names=['Low Impact', 'High Impact']))

# ======================= CELL 26: MODEL COMPARISON ===========================
print("\n" + "=" * 70)
print("  SECTION 23: MODEL COMPARISON")
print("=" * 70)

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'Gradient Boosting'],
    'Accuracy': [rf_acc, xgb_acc, gb_acc],
    'F1 Score': [rf_f1, xgb_f1, gb_f1],
    'ROC AUC': [rf_auc, xgb_auc, gb_auc]
}).round(4)

display(comparison_df) if 'display' in dir() else print(comparison_df)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
w = 0.25
ax.bar(x - w, comparison_df['Accuracy'], w, label='Accuracy', color='#3498db', edgecolor='black')
ax.bar(x, comparison_df['F1 Score'], w, label='F1 Score', color='#e74c3c', edgecolor='black')
ax.bar(x + w, comparison_df['ROC AUC'], w, label='ROC AUC', color='#2ecc71', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
for i in range(3):
    for j, metric in enumerate(['Accuracy', 'F1 Score', 'ROC AUC']):
        val = comparison_df.iloc[i][metric]
        ax.text(i + (j-1)*w, val + 0.02, f'{val:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify best model
best_idx = comparison_df['F1 Score'].idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']
print(f"\n🏆 Best Model (by F1 Score): {best_model_name}")
print("   ✅ Saved: model_comparison.png")

# ======================= CELL 27: CONFUSION MATRICES =========================
print("\n" + "=" * 70)
print("  SECTION 24: CONFUSION MATRICES")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models_data = [
    ('Random Forest', rf_pred),
    ('XGBoost', xgb_pred),
    ('Gradient Boosting', gb_pred)
]

for ax, (name, pred) in zip(axes, models_data):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low Impact', 'High Impact'],
                yticklabels=['Low Impact', 'High Impact'],
                linewidths=1, linecolor='black')
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: confusion_matrices.png")

# ======================= CELL 28: ROC CURVES =================================
print("\n" + "=" * 70)
print("  SECTION 25: ROC CURVES")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 8))

for name, prob, color in [
    ('Random Forest', rf_prob, '#3498db'),
    ('XGBoost', xgb_prob, '#e74c3c'),
    ('Gradient Boosting', gb_prob, '#2ecc71')
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: roc_curves.png")

# ======================= CELL 29: FEATURE IMPORTANCE =========================
print("\n" + "=" * 70)
print("  SECTION 26: FEATURE IMPORTANCE")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
models_imp = [
    ('Random Forest', rf_model),
    ('XGBoost', xgb_model),
    ('Gradient Boosting', gb_model)
]

for ax, (name, model) in zip(axes, models_imp):
    importance = model.feature_importances_
    sorted_idx = np.argsort(importance)
    ax.barh(range(len(sorted_idx)), importance[sorted_idx],
            color=sns.color_palette("viridis", len(sorted_idx)))
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([feature_cols[i] for i in sorted_idx], fontsize=8)
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance — All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: feature_importance.png")

# ======================= CELL 30: CROSS-VALIDATION ===========================
print("\n" + "=" * 70)
print("  SECTION 27: CROSS-VALIDATION (5-Fold Stratified)")
print("=" * 70)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use best model for CV (or all 3)
for name, model in [('Random Forest', rf_model), ('XGBoost', xgb_model), ('Gradient Boosting', gb_model)]:
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1', n_jobs=-1)
    print(f"   {name:20s} — CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})  |  Folds: {cv_scores.round(4)}")

# ======================= CELL 31: ENFORCEMENT PRIORITY ZONES =================
print("\n" + "=" * 70)
print("  SECTION 28: ENFORCEMENT PRIORITY ZONES")
print("=" * 70)

# Rank grid cells by congestion impact
enforcement_zones = df.groupby(['grid_cell', 'police_station']).agg(
    lat=('latitude', 'mean'),
    lng=('longitude', 'mean'),
    avg_cis=('congestion_impact_score', 'mean'),
    total_violations=('id', 'count'),
    unique_vehicles=('vehicle_number', 'nunique'),
    pct_rush_hour=('time_multiplier', lambda x: (x >= 1.3).mean()),
    pct_junction=('junction_multiplier', lambda x: (x > 1).mean()),
    top_violation=('primary_violation', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown')
).reset_index()

# Priority scoring
enforcement_zones['priority_score'] = (
    enforcement_zones['avg_cis'] * 0.4 +
    enforcement_zones['total_violations'] / enforcement_zones['total_violations'].max() * 0.3 +
    enforcement_zones['pct_rush_hour'] * 0.15 +
    enforcement_zones['pct_junction'] * 0.15
).round(4)

# Priority tiers
enforcement_zones['priority_tier'] = pd.qcut(
    enforcement_zones['priority_score'], q=4,
    labels=['Low', 'Medium', 'High', 'Critical']
)

# Top 20 enforcement priority zones
top_enforcement = enforcement_zones.sort_values('priority_score', ascending=False).head(20)

print("\n🚨 TOP 20 ENFORCEMENT PRIORITY ZONES:")
print("-" * 90)
for _, row in top_enforcement.iterrows():
    print(f"  [{row['priority_tier']:8s}] {row['police_station']:25s} | "
          f"Score: {row['priority_score']:.4f} | "
          f"Violations: {int(row['total_violations']):5,} | "
          f"Rush Hr: {row['pct_rush_hour']*100:.0f}% | "
          f"Top: {row['top_violation']}")

# Visualize priority zones
fig, ax = plt.subplots(figsize=(14, 8))
colors_map = {'Critical': 'red', 'High': 'orange', 'Medium': 'yellow', 'Low': 'green'}
for tier in ['Low', 'Medium', 'High', 'Critical']:
    subset = enforcement_zones[enforcement_zones['priority_tier'] == tier]
    ax.scatter(subset['lng'], subset['lat'],
              c=colors_map[tier], label=tier, s=subset['total_violations']*0.3,
              alpha=0.5, edgecolors='black', linewidth=0.2)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Enforcement Priority Zones — Bengaluru', fontsize=14, fontweight='bold')
ax.legend(title='Priority', fontsize=10, markerscale=0.5)
plt.tight_layout()
plt.savefig('enforcement_priority_zones.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: enforcement_priority_zones.png")

# ======================= CELL 32: ENFORCEMENT HEATMAP ========================
print("\n" + "=" * 70)
print("  SECTION 29: ENFORCEMENT PRIORITY INTERACTIVE MAP")
print("=" * 70)

m3 = folium.Map(location=[12.9716, 77.5946], zoom_start=12, tiles='CartoDB positron')

color_map = {'Critical': 'red', 'High': 'orange', 'Medium': 'yellow', 'Low': 'green'}
for _, row in enforcement_zones.iterrows():
    if row['priority_tier'] in ['Critical', 'High']:
        folium.CircleMarker(
            location=[row['lat'], row['lng']],
            radius=max(3, min(row['total_violations'] * 0.05, 15)),
            color=color_map[row['priority_tier']],
            fill=True,
            fill_opacity=0.6,
            popup=(f"<b>Station:</b> {row['police_station']}<br>"
                   f"<b>Priority:</b> {row['priority_tier']}<br>"
                   f"<b>Score:</b> {row['priority_score']:.4f}<br>"
                   f"<b>Violations:</b> {int(row['total_violations']):,}<br>"
                   f"<b>Rush Hour %:</b> {row['pct_rush_hour']*100:.0f}%<br>"
                   f"<b>Top Violation:</b> {row['top_violation']}")
        ).add_to(m3)

m3.save('enforcement_priority_map.html')
print("   ✅ Saved: enforcement_priority_map.html")
try:
    display(m3)
except:
    print("   (Open enforcement_priority_map.html in browser to view)")

# ======================= CELL 33: PRECISION-RECALL CURVES ====================
print("\n" + "=" * 70)
print("  SECTION 30: PRECISION-RECALL CURVES")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 8))

for name, prob, color in [
    ('Random Forest', rf_prob, '#3498db'),
    ('XGBoost', xgb_prob, '#e74c3c'),
    ('Gradient Boosting', gb_prob, '#2ecc71')
]:
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(recall, precision, color=color, linewidth=2.5, label=f'{name} (AP = {ap:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: precision_recall_curves.png")

# ======================= CELL 34: CORRELATION HEATMAP ========================
print("\n" + "=" * 70)
print("  SECTION 31: FEATURE CORRELATION HEATMAP")
print("=" * 70)

corr_cols = [
    'hour', 'day_of_week', 'is_weekend', 'num_violations', 'violation_severity',
    'grid_violation_count', 'station_violation_count', 'daily_grid_violations',
    'hourly_grid_violations', 'vehicle_size_mult', 'density_factor',
    'congestion_impact_score', 'high_impact'
]

fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'fontsize': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("   ✅ Saved: correlation_heatmap.png")

# ======================= CELL 35: FINAL SUMMARY ==============================
print("\n" + "=" * 70)
print("  SECTION 32: FINAL SUMMARY & RECOMMENDATIONS")
print("=" * 70)

best_model_scores = comparison_df.loc[comparison_df['F1 Score'].idxmax()]

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    PARKING INTELLIGENCE — SUMMARY                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  DATASET                                                             ║
║  ─────────                                                           ║
║  • Total violations analyzed: {len(df):>8,}                          ║
║  • Unique grid cells:         {df['grid_cell'].nunique():>8,}        ║
║  ║  • Date range:  {str(df['date'].dropna().min())} to {str(df['date'].dropna().max())}   ║
  ║
║  • Police stations:           {df['police_station'].nunique():>8}    ║
║                                                                      ║
║  HOTSPOT DETECTION                                                   ║
║  ──────────────────                                                  ║
║  • DBSCAN clusters:           {n_clusters:>8}                        ║
║  • HDBSCAN clusters:          {n_hclusters:>8}                       ║
║  • Critical enforcement zones:{len(enforcement_zones[enforcement_zones['priority_tier']=='Critical']):>8,}║
║  • High-priority zones:       {len(enforcement_zones[enforcement_zones['priority_tier']=='High']):>8,}   ║
║                                                                      ║
║  BEST ML MODEL: {best_model_scores['Model']:<20s}                   ║
║  ──────────────────                                                  ║
║  • Accuracy:  {best_model_scores['Accuracy']:.4f}                    ║
║  • F1 Score:  {best_model_scores['F1 Score']:.4f}                    ║
║  • ROC AUC:   {best_model_scores['ROC AUC']:.4f}                    ║
║                                                                      ║
║  KEY INSIGHTS                                                        ║
║  ─────────────                                                       ║
║  1. Most violations occur during Night & Midday hours                ║
║  2. "Wrong Parking" & "No Parking" dominate (~87% of all)            ║
║  3. Main road & junction violations have highest congestion impact   ║
║  4. Larger vehicles (trucks, buses) amplify traffic disruption       ║
║  5. Enforcement should prioritize rush-hour junction hotspots        ║
║                                                                      ║
║  RECOMMENDATIONS                                                     ║
║  ──────────────────                                                  ║
║  • Deploy AI cameras at top 20 Critical priority zones               ║
║  • Increase patrols during Morning Rush (6-10 AM) at junctions      ║
║  • Focus on heavy vehicles in main road violations                   ║
║  • Use CIS-weighted heatmap for dynamic patrol routing               ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# Save all outputs info
print("\n📁 SAVED FILES:")
saved_files = [
    'eda_violation_patterns.png', 'eda_heatmap_hour_day.png',
    'eda_monthly_trend.png', 'eda_vehicle_types.png', 'eda_top_hotspots.png',
    'clustering_comparison.png', 'cis_analysis.png',
    'violation_heatmap.html', 'cis_heatmap.html',
    'model_comparison.png', 'confusion_matrices.png',
    'roc_curves.png', 'feature_importance.png',
    'precision_recall_curves.png', 'correlation_heatmap.png',
    'enforcement_priority_zones.png', 'enforcement_priority_map.html'
]
for f in saved_files:
    print(f"   ✅ {f}")

print("\n" + "=" * 70)
print("  PIPELINE COMPLETE!")
print("=" * 70)


In [4]:
# ======================= CELL 36: EXPORT PRODUCTION MODEL ====================
print("\n" + "=" * 70)
print("  SECTION 33: EXPORT PRODUCTION BACKEND MODEL")
print("=" * 70)

import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

print("Training production deployment pipeline...")

# Features that the FastAPI backend will send in real-time
deploy_features = ['hour', 'day_of_week', 'is_weekend', 'vehicle_type']
X_deploy = df[deploy_features]
y_deploy = df['congestion_impact_score']

numeric_features = ["hour", "day_of_week", "is_weekend"]
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ["vehicle_type"]
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=100, max_depth=6, random_state=42))
])

production_pipeline.fit(X_deploy, y_deploy)

# Save to Kaggle's output directory
joblib.dump(production_pipeline, '/kaggle/working/gridlock_pipeline_v_new.pkl')
print("   ✅ Model saved to: /kaggle/working/gridlock_pipeline_v_new.pkl")
print("   📥 Download from the 'Output' tab on the right sidebar!")



  SECTION 33: EXPORT PRODUCTION BACKEND MODEL
Training production deployment pipeline...
   ✅ Model saved to: /kaggle/working/gridlock_pipeline_v_new.pkl
   📥 Download from the 'Output' tab on the right sidebar!


In [5]:
import joblib, os
model = joblib.load('/kaggle/working/gridlock_pipeline_v_new.pkl')
print(f"Model type: {type(model)}")
print(f"File size: {os.path.getsize('/kaggle/working/gridlock_pipeline_v_new.pkl') / 1024:.1f} KB")
print("✅ Model is valid and saved!")


Model type: <class 'sklearn.pipeline.Pipeline'>
File size: 423.8 KB
✅ Model is valid and saved!
